In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'

In [3]:
arquivos_csv = list(RAW_DIR.glob('*.csv'))

def carregar_dados_csv():
    dados = {}

    arquivos = {
        'orders':'olist_orders_dataset.csv',
        'customers':'olist_customers_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'order_payments': 'olist_order_payments_dataset.csv',
        'order_reviews': 'olist_order_reviews_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv'
    }

    for nome, arquivo in arquivos.items():
        caminho = RAW_DIR / arquivo
        if caminho.exists():
            try:
                dados[nome] = pd.read_csv(caminho)
            except Exception as e:
                print(f"Erro ao carregar o arquivo {caminho}: {e}")
        else:
            print(f"Arquivo {caminho} não encontrado.")
    return dados

In [4]:
dados = carregar_dados_csv()

In [5]:
def visao_geral(df, nome):
    print(f"Visão geral do DataFrame: {nome.upper()}")
    print("-" * 50)
    print("Dimensões:", df.shape)
    print("\nTipos de dados:")
    print(df.dtypes)
    print("\nValores ausentes:")
    print(df.isnull().sum())
    print("\nEstatísticas descritivas:")
    print(df.describe(include='all'))
    print("\nValores únicos por coluna:")
    for coluna in df.columns:
        print(f"{coluna}: {df[coluna].nunique()} valores únicos")
    print("-" * 50)

In [6]:
for nome, df in dados.items():
    visao_geral(df, nome)

Visão geral do DataFrame: ORDERS
--------------------------------------------------
Dimensões: (99441, 8)

Tipos de dados:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Valores ausentes:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Estatísticas descritivas:
                                order_id                       customer_id  \
count                              99441                             99441   
unique                             99441        

In [7]:
print("Análise de relacionamento entre tabelas")

relacionamentos = {
    'orders': {
        'pk': 'order_id',
        'fks': ['customer_id']
    },

    'customers': {
        'pk': 'customer_id',
        'fks': []
    },

    'products': {
        'pk': 'product_id',
        'fks': []
    },

    'order_items': {
        'pk': 'order_item_id',
        'fks': ['order_id', 'product_id', 'seller_id']
    },

    'order_payments': {
        'pk': None,
        'fks': ['order_id']
    },

    'order_reviews': {
        'pk': None,
        'fks': ['order_id']
    },

    'sellers': {
        'pk': 'seller_id',
        'fks': []
    }
}

for tabela, info in relacionamentos.items():
    if tabela in dados:
        print(f"Tabela: {tabela.upper()}")
        print(f"Chave primária: {info['pk']}")
        print(f"Chaves estrangeiras: {info['fks']}")
        print("-" * 50)

Análise de relacionamento entre tabelas
Tabela: ORDERS
Chave primária: order_id
Chaves estrangeiras: ['customer_id']
--------------------------------------------------
Tabela: CUSTOMERS
Chave primária: customer_id
Chaves estrangeiras: []
--------------------------------------------------
Tabela: PRODUCTS
Chave primária: product_id
Chaves estrangeiras: []
--------------------------------------------------
Tabela: ORDER_ITEMS
Chave primária: order_item_id
Chaves estrangeiras: ['order_id', 'product_id', 'seller_id']
--------------------------------------------------
Tabela: ORDER_PAYMENTS
Chave primária: None
Chaves estrangeiras: ['order_id']
--------------------------------------------------
Tabela: ORDER_REVIEWS
Chave primária: None
Chaves estrangeiras: ['order_id']
--------------------------------------------------
Tabela: SELLERS
Chave primária: seller_id
Chaves estrangeiras: []
--------------------------------------------------


In [ ]:
#Resumo das tabelas

lista = []

for name, df in dados.items():
    lista.append({
        'Tabela': name,
        'Registros': len(df),  
        'Colunas': len(df.columns),
        'memoria (MB)': df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

resumo = pd.DataFrame(lista)
print(resumo.to_string(index=False))


Resumo das tabelas:
        Tabela  Registros  Colunas  Memoria (MB)
        orders    99441.0      8.0     52.937277
     customers    99441.0      5.0     26.586405
      products    32951.0      9.0      6.296564
   order_items   112650.0      7.0     35.989649
order_payments   103886.0      5.0     16.229413
 order_reviews    99224.0      7.0     39.124777
       sellers     3095.0      4.0      0.588103


In [9]:
resumo.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'resumo_tabelas.csv', index=False)
print("\nResumo das tabelas salvo em 'data/processed/resumo_tabelas.csv'")


Resumo das tabelas salvo em 'data/processed/resumo_tabelas.csv'
